In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
from transformers import AutoModelForMultipleChoice, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import torch
import torch.nn as nn

MAX_LEN     = 256
BATCH_SIZE  = 16
EPOCHS      = 5
LR          = 2e-4
LABEL_MAP   = {'A':0,'B':1,'C':2,'D':3,'E':4}
INV_LABEL_MAP = {0 :'A',1:'B',2:'C',3:'D',4:'E'}
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
options = ['A','B','C','D','E']

In [4]:
model = AutoModelForMultipleChoice.from_pretrained("roberta-base")
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
print(sum(p.numel() for p in model.parameters() if p.requires_grad))
num_encoder_layers = len(model.roberta.encoder.layer)
print(f"Number of encoder layers: {num_encoder_layers}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.pooler.dense.weight     | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

124646401
Number of encoder layers: 12


In [5]:
# Freeze all parameters first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last 4 encoder layers
num_layers = len(model.roberta.encoder.layer)
for i in range(num_layers - 2, num_layers):
    for param in model.roberta.encoder.layer[i].parameters():
        param.requires_grad = True

# Also unfreeze the pooler and classifier
if hasattr(model.roberta, 'pooler'):
    for param in model.roberta.pooler.parameters():
        param.requires_grad = True

if hasattr(model, 'classifier'):
    for param in model.classifier.parameters():
        param.requires_grad = True

print(f"Number of trainable parameters after freezing: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6}M")

Number of trainable parameters after freezing: 14.767105M


In [6]:
class MCQDataset(Dataset):
    def __init__(self, df, has_labels=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = MAX_LEN
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        prompt = row['prompt']
        input_ids = []
        attention_mask = []

        for opt in options:
            opt_text = row[opt]
            encoding = self.tokenizer(prompt, opt_text, max_length=self.max_length,
                                  truncation = True, padding='max_length', return_tensors='pt')
            input_ids.append(encoding['input_ids'])
            attention_mask.append(encoding['attention_mask'])

        input_ids = torch.stack(input_ids)
        attention_mask = torch.stack(attention_mask)

        if self.has_labels:
            labels = torch.tensor(LABEL_MAP[row['answer']])
            return {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels
            }
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'id': row['id']
        } 

In [7]:
tr_df, val_df = train_test_split(
    train, test_size=0.2, random_state=42, stratify=train["answer"]
)
train_dataset = MCQDataset(tr_df, has_labels=True)
val_dataset = MCQDataset(val_df, has_labels=True)
test_dataset = MCQDataset(test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, shuffle=False)

In [8]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS
)

In [9]:
def run_epoch(model, loader, optimizer, scheduler, device, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    correct = 0

    with torch.set_grad_enabled(train):
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            if train:
                optimizer.zero_grad()

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss.mean()

            if train:
                loss.backward()
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()

            preds = outputs.logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)

    return avg_loss, accuracy

In [10]:
import time
model = nn.DataParallel(model)
model.to(DEVICE)
for epoch in range(EPOCHS):

    start_time = time.time()

    train_loss, train_acc = run_epoch(model=model, loader=train_loader, optimizer=optimizer,
        scheduler=scheduler, device=DEVICE, train=True)

    train_end_time = time.time()
    train_time = train_end_time - start_time

    val_loss, val_acc = run_epoch(model=model, loader=val_loader, optimizer=None,
        scheduler=None, device=DEVICE, train=False )

    val_end_time = time.time()
    val_time = val_end_time - train_end_time

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Train Time: {train_time:.2f}s | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
        f"Val Time: {val_time:.2f}s"
    )

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 1/5 | Train Loss: 1.5242 | Train Acc: 0.2906 | Train Time: 83.35s | Val Loss: 0.8612 | Val Acc: 0.6650 | Val Time: 14.65s
Epoch 2/5 | Train Loss: 0.6298 | Train Acc: 0.7588 | Train Time: 88.15s | Val Loss: 0.2220 | Val Acc: 0.9450 | Val Time: 14.63s
Epoch 3/5 | Train Loss: 0.2683 | Train Acc: 0.9000 | Train Time: 87.47s | Val Loss: 0.0742 | Val Acc: 0.9900 | Val Time: 14.53s
Epoch 4/5 | Train Loss: 0.1314 | Train Acc: 0.9494 | Train Time: 87.01s | Val Loss: 0.0400 | Val Acc: 0.9925 | Val Time: 14.56s
Epoch 5/5 | Train Loss: 0.1108 | Train Acc: 0.9587 | Train Time: 87.13s | Val Loss: 0.0331 | Val Acc: 0.9900 | Val Time: 14.52s


In [11]:
import torch.nn.functional as F
model.to('cuda')
model.eval()
with torch.no_grad():
    sample   = train_dataset[923]
    ids      = sample["input_ids"].unsqueeze(0).to(DEVICE)
    attn     = sample["attention_mask"].unsqueeze(0).to(DEVICE)
    logits   = model(input_ids=ids, attention_mask=attn).logits
    probs    = F.softmax(logits, dim=-1)
    top3_p, top3_i = torch.topk(probs, k=3, dim=-1)
    options  = ['A','B','C','D','E']
    print("Top-3 guesses:", [options[i] for i in top3_i[0].tolist()])
    print("Confidences:  ", top3_p[0].tolist())
    print(f"Actual Answer: {INV_LABEL_MAP[sample['labels'].item()]}")


Top-3 guesses: ['C', 'D', 'B']
Confidences:   [0.9999983310699463, 1.2510818123701029e-06, 1.9927861671931169e-07]
Actual Answer: C


In [12]:
model.eval()

all_ids         = []
all_top_n_labels = []
all_top_n_probs  = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        ids            = batch["id"].tolist()

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask)
        probs   = F.softmax(outputs.logits, dim=-1)

        top_probs, top_idx = torch.topk(probs, k=3, dim=-1)

        for i, qid in enumerate(ids):
            labels_i = [INV_LABEL_MAP[j.item()] for j in top_idx[i]]
            probs_i  = [round(p.item(), 4) for p in top_probs[i]]

            all_ids.append(qid)
            all_top_n_labels.append(labels_i)
            all_top_n_probs.append(probs_i)

In [13]:
submission = pd.DataFrame({
    "ID":        all_ids,
    "Prediction": [" ".join(labels) for labels in all_top_n_labels],
})
submission.to_csv("submission.csv", index=False)
print(submission.head())

   ID Prediction
0   1      A C B
1   2      B E A
2   3      B E D
3   4      E C A
4   5      C A B


In [14]:
# 1. Save the model's state dictionary
torch.save(model.state_dict(), 'roberta_sped_14m.pt')
print("Model state dictionary saved as 'roberta_sped_14m.pt'")

Model state dictionary saved as 'roberta_sped_14m.pt'
